In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspsark-pilot')
    .master('local[*]')
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 19:35:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema_string = StructType([
    StructField('employee_id', StringType(), False),
    StructField('department_id', StringType(), False),
    StructField('name', StringType(), False),
    StructField('age', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('salary', StringType(), True),
    StructField('hire_date', StringType(), True)
])

data_1 = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"]
]
    
data_2 = [
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

from pyspark.sql.functions import col, to_date

def cast_df(df):
    return df.select(
        'employee_id', 'department_id', 'name', 
        col('age').cast('integer'), 
        'gender', 
        col('salary').cast('double'), 
        to_date('hire_date', 'yyyy-MM-dd').alias('hire_date')
    )

emp_1_df = cast_df(spark.createDataFrame(data=data_1, schema=schema_string))
emp_2_df = cast_df(spark.createDataFrame(data=data_2, schema=schema_string))

emp_1_df.show()
emp_2_df.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|        001|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|        002|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|        004|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|        006|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
+-----------+-------------+-------------+---+------+-------+----

In [3]:
# Union

emp_df = emp_1_df.union(emp_2_df)

emp_df.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|        001|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|        002|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|        004|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|        006|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
|        011|          104|   David Park| 38|  Male|65000.0|2015

In [4]:
# Ordering

from pyspark.sql.functions import col, asc, desc

emp_df.orderBy(col('salary').desc()).show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|        013|          106|    Brian Kim| 45|  Male|75000.0|2011-07-01|
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|        011|          104|   David Park| 38|  Male|65000.0|2015-11-01|
|        015|          106|  Michael Lee| 37|  Male|63000.0|2014-09-30|
|        019|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|
|        005|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|        009|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|        017|          105|  George Wang| 34|  Male|57000.0|2016-03-15|
|        003|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|        012|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018

In [5]:
# Aggregating

from pyspark.sql.functions import count

emp_df.groupBy('department_id').agg(count('employee_id').alias('employee_count')).orderBy(col('employee_count').desc()).show()

+-------------+--------------+
|department_id|employee_count|
+-------------+--------------+
|          102|             4|
|          103|             4|
|          101|             3|
|          104|             3|
|          105|             2|
|          106|             2|
|          107|             2|
+-------------+--------------+



In [6]:
# Having

from pyspark.sql.functions import avg

emp_df.groupBy('department_id').agg(avg('salary').alias('avg_salary')).where(col('avg_salary') > 50000).show()

+-------------+----------+
|department_id|avg_salary|
+-------------+----------+
|          101|   55000.0|
|          102|   51750.0|
|          103|   58000.0|
|          104|   54000.0|
|          105|   55500.0|
|          106|   69000.0|
+-------------+----------+

